### **Student Information**
GitHub ID:  triababtista

Kaggle name: Babtista Putri Grahani

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

***

# **Project Report**

**Task:** 6-class emotion recognition (anger, disgust, fear, joy, sadness, surprise) on noisy social-media posts. Metric: macro F1. Data: 3,360 labelled training posts (3,294 after cleaning) and 1,440 test posts.

**Final solution in one line:** a blend (50/50, weight chosen on out-of-fold macro F1) of (a) Logistic Regression on TF-IDF + frozen transformer features and (b) the average of three fine-tuned transformers (one per backbone), all evaluated on the same 5 folds.

| Model | Out-of-fold macro F1 |
| --- | --- |
| LR on TF-IDF + frozen features | 0.6346 |
| Fine-tuned `roberta-base-go_emotions` | 0.6113 |
| Fine-tuned `emotion-english-distilroberta-base` (Hartmann) | 0.6046 |
| Fine-tuned `twitter-roberta-base-emotion-multilabel-latest` | 0.5830 |
| Average of the 3 fine-tuned runs | 0.6346 |
| **Blend: 0.5 x fine-tuned average + 0.5 x LR** | **0.6452** |

My Kaggle Private Leaderboard result is in the screenshot at the top of this notebook.

---

## 1. Model Development (10 pts Required)

### 1.1 Preprocessing Steps

The posts in `final_posts.json` are much noisier than the clean tweets used in the lab: some are machine-translated or in other languages, some contain placeholders (`[NAME]`, `@<username>`, `##URL##`), and some are exact duplicates. My steps were:

* **Merge** posts, labels (`emotion.csv`) and the train/test split (`data_identification.csv`) on `post_id`.
* **Duplicate handling (train only):**
    * same text with different labels -> drop all copies (the label is unreliable)
    * same text with the same label -> keep one copy
    * the test set is never touched, because every test post needs a prediction.
    * Result: 3,360 -> 3,294 training posts.
* **Two cleaning functions**, because TF-IDF and transformers need different input:
    * `clean_tfidf` (heavy): placeholders and links become word tokens (`nametoken`, `urltoken`, `usertoken`), style cues (ALL CAPS, elongated letters, `!!`, `...`) are added as extra tokens, emojis are kept and also written out as English words, text is lower-cased and letter repetitions are shortened (`soooo` -> `soo`). Punctuation is kept, and no stop-word removal or stemming is applied, since these carry emotion.
    * `clean_transformer` (light): only links -> `link` and usernames -> `@user`, so the pretrained tokenizers see natural text.
* **Label encoding and folds:** labels are encoded to 0-5 and 5 folds are built with `StratifiedGroupKFold`. Duplicate texts are used as groups, so a duplicate can never sit in both the training and validation part of a fold. **Every model uses the same folds**, which makes the out-of-fold predictions comparable and blendable.

### 1.2 Feature Engineering Steps

* **TF-IDF on words (1-2 grams) and characters (2-5 grams, `char_wb`)**, sublinear term frequency, `min_df=2`. Character n-grams help with typos, slang and badly translated text where word vocabularies fail. The vectoriser is fitted **inside each fold**, so the validation vocabulary never leaks.
* **Frozen transformer features (nothing is trained):** for both `SamLowe/roberta-base-go_emotions` and `j-hartmann/emotion-english-distilroberta-base` I take (1) the mean-pooled last hidden layer (L2-normalised) and (2) the model's own **emotion-head logits** (28 and 7 outputs), standardised with training statistics only. This gives 1,571 dense features that are stacked next to the TF-IDF matrix.
* **Fine-tuned transformer probabilities** (see 1.3) are used as a second, separate view and blended with the LR output.

### 1.3 Explanation of Your Model

**Model 1: Logistic Regression** on `[TF-IDF word + char | frozen embeddings + emotion logits]`, `C=1.0`, `class_weight="balanced"` (the metric is macro F1, so every class should count equally). OOF macro F1 = 0.6346.

**Model 2: fine-tuned transformers, averaged.** One run per backbone (`go_emotions`, Hartmann DistilRoBERTa, Twitter-RoBERTa), same seed (42), same 5 folds. Settings: 3 epochs, lr 2e-5, batch size 32, max length 128, AdamW (weight decay 0.01), linear schedule with 10% warm-up, gradient clipping 1.0, mixed precision, and class-weighted cross-entropy. The classification head is re-initialised for 6 classes. Only the **last epoch** is used, so the out-of-fold score is not optimistically picked. For each run the 5 fold models give out-of-fold predictions for the training posts, and their mean gives the test prediction. The 3 runs are then averaged with equal weights. The leave-one-out check (0.6223 / 0.6245 / 0.6288 when removing one run) shows that no single run is dragging the average down.

**Final blend:** `p = w * (fine-tuned average) + (1 - w) * LR`, with `w` scanned from 0 to 1 on the out-of-fold predictions. The best was `w = 0.5` (OOF macro F1 0.6452), and the curve is flat between 0.4 and 0.7 (0.644-0.645), so the choice is not a lucky pick. The test prediction takes the argmax of the blended probabilities.

**Error analysis:** `joy` is the easiest class (F1 0.78) and `disgust` is the hardest (F1 0.49), mostly confused with `anger` (105 disgust posts predicted as anger and 79 the other way round). This is expected, because the two emotions use similar vocabulary in short posts.

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

I used an AI assistant at every stage and kept only the suggestions that were cheap, safe against leakage, and helped the cross-validation score. Everything was judged on out-of-fold macro F1, not on the public leaderboard.

| Step | What I asked / what the AI suggested | What I decided and why |
| --- | --- | --- |
| 1. Strategy | I explain the task and give my rough plan about data preprocessing and the strategy to use the best practice from Master's notebook (TF-IDF + LinearSVM). The AI inspected the data and adjusted the plan (6 classes, noisy and translated text, placeholders, duplicates, repetition loops), then proposed the baseline draft. | **Used:** the structure of the plan, the placeholder tokens, keeping punctuation/emojis, TF-IDF word + char features, LR, and CV on macro F1. **Not used:** the self-trained Word2Vec/Doc2Vec ideas (near chance on this little data) and the extra meta-features. |
| 2. Baseline | The AI drafted the baseline pipeline. | I **reviewed and added the duplicate removal** (same text and different labels -> drop all, same text and same label -> keep one) and made the folds group-aware. |
| 3. Next strategy | I asked what to try next. The AI listed, cheapest first: use the model's emotion head, add a second/third embedding view, fine-tune instead of freezing, blend separate models, run another seed. | **Used:** the emotion-head logits as features (they were being thrown away by `AutoModel`), and a second view from the Hartmann model, which helped on CV. Other extra frozen views did not help, so I kept only the useful ones. **Used:** fine-tuning, with a blend with LR whose weight is chosen on out-of-fold predictions. |
| 4. Further improvement | I asked again what else could improve the score. The AI suggested a larger model (`roberta-large`), averaging more fine-tuned runs, cleaning noisy labels, stacking, and tuning epochs/learning rate. | **Used:** averaging fine-tuned runs from different backbones (cheap, about 4 minutes each, and it makes the fine-tuned part less noisy) and the weighted blend. **Rejected:** `roberta-large` (long training and unstable on about 3,300 posts, with an uncertain gain), label cleaning (it could also remove hard but correct posts), stacking (small gain, risk of leakage), and epoch/lr tuning (expected gain is tiny). |

**Reason for the final choice:** I preferred a simple, faster, reproducible ensemble (LR + three fine-tuned base models) over a riskier large model. Its improvement over any single model is visible in the out-of-fold scores (0.6452 vs 0.6346 for the best single component).

---

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

### Setup: imports, file locations, settings

In [ ]:
import json, re, subprocess, sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
from torch.utils.data import DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import normalize
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          get_linear_schedule_with_warmup)

try:
    import emoji
except ImportError:                       # fresh Colab: install it
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "emoji"])
    import emoji

# Folders. Kaggle: data under /kaggle/input, results to /kaggle/working.
#          Colab:  files you uploaded to /content, results to /content/outputs.
if Path("/kaggle/input").exists():
    INPUT_ROOT, OUTPUT_DIR = Path("/kaggle/input"), Path("/kaggle/working")
else:
    INPUT_ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
    OUTPUT_DIR = INPUT_ROOT / "outputs"
    OUTPUT_DIR.mkdir(exist_ok=True)

def input_file(name):
    """Find a data file anywhere under /kaggle/input."""
    hits = sorted(INPUT_ROOT.rglob(name))
    if not hits:
        raise FileNotFoundError(f"{name} not found - add the dataset via 'Add Input'")
    return hits[0]

# Settings
GO_MODEL   = "SamLowe/roberta-base-go_emotions"                          # frozen features + fine-tuned
HART_MODEL = "j-hartmann/emotion-english-distilroberta-base"             # frozen features + fine-tuned
TW_MODEL = "cardiffnlp/twitter-roberta-base-emotion-multilabel-latest"   # trained on tweets, 11 emotions
MAX_LEN, BATCH_SIZE, EPOCHS, LR_FT = 128, 32, 3, 2e-5
SEED, N_FOLDS = 42, 5

# Fine-tuning runs: ONE RUN PER BACKBONE (seed 42 for all), all on the same 5 folds. Their predictions are averaged.
# "legacy" = go, seed 42: the files saved as ft_fold<k>_*.npy by the first single-run notebook.
# If a run's files already exist it is NOT retrained. Optional per-run keys: lr, epochs (defaults: LR_FT, EPOCHS).
RUNS = [
    dict(tag="legacy",   model=GO_MODEL,   seed=42),
    dict(tag="hart_s42", model=HART_MODEL, seed=42),
    dict(tag="tw_s42",   model=TW_MODEL,   seed=42),
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print("device:", device, "| outputs:", OUTPUT_DIR)

### 1a. Load and merge posts, labels and the train/test split

In [ ]:
posts = pd.DataFrame(json.load(open(input_file("final_posts.json"), encoding="utf-8")))
labels = pd.read_csv(input_file("emotion.csv"))
split = pd.read_csv(input_file("data_identification.csv"))
sample_sub = pd.read_csv(input_file("sampleSubmission.csv"))

data = posts.merge(split, on="post_id").merge(labels, on="post_id", how="left")
train = data[data["set"] == "train"].reset_index(drop=True)
test = data[data["set"] == "test"].reset_index(drop=True)

### 1b. Duplicates (train only): same text + different labels -> drop all copies,

In [ ]:
# same text + same label -> keep one. Test is untouched (needs a prediction each).
train["dup_key"] = train["text"].str.lower().str.strip()
n_labels = train.groupby("dup_key")["label"].transform("nunique")
train = train[n_labels == 1].drop_duplicates("dup_key").reset_index(drop=True)
print("train:", train.shape, "test:", test.shape)

### 1c. Two cleaning functions

In [ ]:
def clean_tfidf(text):
    """Heavy cleaning for TF-IDF: placeholders -> words, style cues -> tokens, emojis -> names."""
    text = str(text)
    text = re.sub(r"\[NAME\]", " nametoken ", text, flags=re.I)
    text = re.sub(r"##URL##|https?://\S+", " urltoken ", text, flags=re.I)
    text = re.sub(r"@<username>|@\w+", " usertoken ", text, flags=re.I)
    cues = []                                          # extra words that capture writing style
    if any(len(w) > 1 and w.isupper() for w in text.split()): cues.append("allcapstoken")
    if re.search(r"(.)\1{2,}", text):                  cues.append("elongatedtoken")
    if "!!" in text:                                   cues.append("multiexcltoken")
    if "..." in text:                                  cues.append("ellipsistoken")
    text = emoji.replace_emoji(                        # keep the emoji and add its English name
        text, replace=lambda ch, d: f" {ch} {d['en'].strip(':').replace('_', ' ')} ")
    text = re.sub(r"(.)\1{2,}", r"\1\1", text.lower()) # soooo -> soo
    return re.sub(r"\s+", " ", text).strip() + " " + " ".join(cues)

def clean_transformer(text):
    """Light cleaning for transformers: only simplify links and usernames."""
    text = re.sub(r"##URL##|https?://\S+", "link", str(text), flags=re.I)
    text = re.sub(r"@<username>|@\w+", "@user", text, flags=re.I)
    return re.sub(r"\s+", " ", text).strip()

train["clean"] = train["text"].map(clean_tfidf)
test["clean"] = test["text"].map(clean_tfidf)

### 1d. Encode labels and build 5 folds. Duplicates stay in the same fold.

In [ ]:
#     Both models use the SAME folds, so their out-of-fold predictions can be blended.
classes = np.sort(train["label"].unique())
n_classes = len(classes)
y = train["label"].map({c: i for i, c in enumerate(classes)}).values
folds = list(StratifiedGroupKFold(N_FOLDS, shuffle=True, random_state=1)
             .split(train["clean"], y, groups=train["dup_key"]))

## 2. Feature Engineering Steps

### 2a. TF-IDF on words (1-2 grams) and characters (2-5 grams).

In [ ]:
#     Built inside each fold so the vocabulary only comes from training rows.
def make_tfidf():
    return FeatureUnion([
        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True,
                                 token_pattern=r"(?u)\b\w+\b|[^\w\s]", lowercase=False)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True,
                                 max_features=300000, lowercase=False)),
    ])

### 2b. Frozen transformer features (nothing is trained here):

In [ ]:
#     mean-pooled last-layer embedding + the model's own emotion-head logits.
@torch.no_grad()
def frozen_features(model_id, texts, bs=64):
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSequenceClassification.from_pretrained(model_id).to(device).eval()
    texts = [clean_transformer(t) for t in texts]
    embs, logits = [], []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i + bs], padding=True, truncation=True,
                  max_length=MAX_LEN, return_tensors="pt").to(device)
        out = model(**enc, output_hidden_states=True)
        m = enc["attention_mask"].unsqueeze(-1).float()
        embs.append(((out.hidden_states[-1] * m).sum(1) / m.sum(1)).cpu().numpy())
        logits.append(out.logits.float().cpu().numpy())
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return normalize(np.vstack(embs)), np.vstack(logits)

all_texts = list(train["text"]) + list(test["text"])
n_tr = len(train)
blocks = []
for model_id in [GO_MODEL, HART_MODEL]:
    emb, logits = frozen_features(model_id, all_texts)
    mu, sd = logits[:n_tr].mean(0), logits[:n_tr].std(0) + 1e-6     # standardise with train stats only
    blocks += [emb, (logits - mu) / sd / np.sqrt(logits.shape[1])]
dense = np.hstack(blocks).astype(np.float32)
train_dense, test_dense = dense[:n_tr], dense[n_tr:]
print("dense features:", train_dense.shape)

## 3. Model Implementation Steps

### 3a. Model 1 - Logistic Regression on [TF-IDF + frozen features].


In [ ]:
#     Class-balanced weights because the metric is macro F1.
def fit_lr(X, y_):
    return LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000).fit(X, y_)

lr_oof = np.zeros((n_tr, n_classes), dtype=np.float32)     # out-of-fold probabilities
for k, (tr, va) in enumerate(folds):
    vec = make_tfidf()
    X_tr = sp.hstack([vec.fit_transform(train["clean"].iloc[tr]), train_dense[tr]], format="csr")
    X_va = sp.hstack([vec.transform(train["clean"].iloc[va]), train_dense[va]], format="csr")
    lr_oof[va] = fit_lr(X_tr, y[tr]).predict_proba(X_va)
    print(f"LR fold {k} done")
print(f"LR out-of-fold macro F1: {f1_score(y, lr_oof.argmax(1), average='macro'):.4f}")

# Final LR: train on all labelled posts, predict test
vec = make_tfidf()
X_full = sp.hstack([vec.fit_transform(train["clean"]), train_dense], format="csr")
X_test = sp.hstack([vec.transform(test["clean"]), test_dense], format="csr")
lr_test = fit_lr(X_full, y).predict_proba(X_test)

### 3b. Model 2 - fine-tune transformers: one run per backbone, one model per fold each, then average the runs.

In [ ]:
#     Weighted cross-entropy (macro F1), last epoch only (keeps OOF score honest).
def collate(batch):                       # pad per batch (faster than padding to 128)
    pad = tokenizer.pad({"input_ids": [b[0] for b in batch]}, return_tensors="pt")
    return pad["input_ids"], pad["attention_mask"], torch.tensor([b[1] for b in batch])

@torch.no_grad()
def predict(model, id_lists, bs=128):
    model.eval()
    order = np.argsort([len(x) for x in id_lists])         # similar lengths together
    out = np.zeros((len(id_lists), n_classes), dtype=np.float32)
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        ids, mask, _ = collate([(id_lists[j], 0) for j in idx])
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
        out[idx] = torch.softmax(logits.float(), -1).cpu().numpy()
    return out

def train_fold(k, model_id, seed, lr=LR_FT, epochs=EPOCHS):
    tr, va = folds[k]
    torch.manual_seed(seed + k)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id, num_labels=n_classes, ignore_mismatched_sizes=True).to(device)
    loader = DataLoader([(train_ids[i], int(y[i])) for i in tr],
                        batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
    counts = np.bincount(y[tr], minlength=n_classes)
    loss_fn = torch.nn.CrossEntropyLoss(
        weight=torch.tensor(counts.sum() / (n_classes * counts), dtype=torch.float32, device=device))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = epochs * len(loader)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    scaler = torch.amp.GradScaler(enabled=use_amp)
    for epoch in range(1, epochs + 1):
        model.train()
        for ids, mask, labs in loader:
            ids, mask, labs = ids.to(device), mask.to(device), labs.to(device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(input_ids=ids, attention_mask=mask).logits
            loss = loss_fn(logits.float(), labs)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
        val_p = predict(model, [train_ids[i] for i in va])
        print(f"  fold {k} epoch {epoch}: val macro F1 {f1_score(y[va], val_p.argmax(1), average='macro'):.4f}")
    test_p = predict(model, test_ids)
    del model, opt
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return val_p, test_p

def fpath(tag, k, kind):
    """File for run `tag`, fold k (kind = 'val' or 'test'). 'legacy' keeps the earlier file names."""
    name = f"ft_fold{k}_{kind}.npy" if tag == "legacy" else f"ft_{tag}_fold{k}_{kind}.npy"
    return OUTPUT_DIR / name

ft_runs = {}                                              # tag -> (out-of-fold probs, averaged test probs)
for run in RUNS:
    tag, model_id, seed = run["tag"], run["model"], run["seed"]
    todo = [k for k in range(N_FOLDS) if not (fpath(tag, k, "val").exists() and fpath(tag, k, "test").exists())]
    if todo:                                              # finished folds are skipped on re-run
        print(f"=== run {tag}: {model_id}, seed {seed}, lr {run.get('lr', LR_FT)}, epochs {run.get('epochs', EPOCHS)}, folds {todo} ===")
        tokenizer = AutoTokenizer.from_pretrained(model_id)          # used by collate() / predict()
        train_ids = tokenizer([clean_transformer(t) for t in train["text"]], truncation=True, max_length=MAX_LEN)["input_ids"]
        test_ids = tokenizer([clean_transformer(t) for t in test["text"]], truncation=True, max_length=MAX_LEN)["input_ids"]
        for k in todo:
            v, t = train_fold(k, model_id, seed, run.get("lr", LR_FT), run.get("epochs", EPOCHS))
            np.save(fpath(tag, k, "val"), v); np.save(fpath(tag, k, "test"), t)
    oof = np.zeros((n_tr, n_classes), dtype=np.float32)
    tst = np.zeros((len(test), n_classes), dtype=np.float32)
    for k, (tr, va) in enumerate(folds):
        oof[va] = np.load(fpath(tag, k, "val"))
        tst += np.load(fpath(tag, k, "test")) / N_FOLDS   # average the 5 fold models on test
    ft_runs[tag] = (oof, tst)
    print(f"  {tag:<9} alone, out-of-fold macro F1: {f1_score(y, oof.argmax(1), average='macro'):.4f}")

# Average the runs with equal weight
ft_oof = np.mean([o for o, _ in ft_runs.values()], axis=0)
ft_test = np.mean([t for _, t in ft_runs.values()], axis=0)
print(f"Average of {len(ft_runs)} runs, out-of-fold macro F1: {f1_score(y, ft_oof.argmax(1), average='macro'):.4f}")
if len(ft_runs) > 2:                                      # does one weak run drag the average down?
    for tag in ft_runs:
        rest = np.mean([o for t_, (o, _) in ft_runs.items() if t_ != tag], axis=0)
        print(f"  leaving out {tag:<9}: {f1_score(y, rest.argmax(1), average='macro'):.4f}")


### 3c. Blend: w * (average of fine-tuned runs) + (1 - w) * LR, with w picked on out-of-fold macro F1

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

scores = {w: f1_score(y, (w * ft_oof + (1 - w) * lr_oof).argmax(1), average="macro")
          for w in np.round(np.linspace(0, 1, 11), 1)}
w_best = max(scores, key=scores.get)
print({w: round(s, 4) for w, s in scores.items()})     # a flat curve around w_best means it is not a lucky pick
print(f"Best blend weight on fine-tuned model: {w_best} (OOF macro F1 {scores[w_best]:.4f})")
blend_oof = w_best * ft_oof + (1 - w_best) * lr_oof
blend_test = w_best * ft_test + (1 - w_best) * lr_test
pred_oof = blend_oof.argmax(1)
print(classification_report(y, pred_oof, target_names=classes, digits=3))

cm = confusion_matrix(y, pred_oof)                      # rows = true, cols = predicted
cm_norm = cm / cm.sum(1, keepdims=True)                 # each row sums to 1 (= recall per class)
print(pd.DataFrame(cm, index=classes, columns=classes))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, M, title, fmt in [(axes[0], cm, "Counts", "{:d}"), (axes[1], cm_norm, "Row-normalized (recall)", "{:.2f}")]:
    ax.imshow(M, cmap="Blues")
    ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=45, ha="right")
    ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, fmt.format(M[i, j]), ha="center", va="center",
                    color="white" if M[i, j] > M.max() * 0.6 else "black")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()

print("Predicted class share, test:", dict(zip(classes, np.round(np.bincount(blend_test.argmax(1), minlength=n_classes) / len(blend_test), 3))))

### 3d. Write submission.csv (columns: id, label)

In [ ]:
pred = dict(zip(test["post_id"].astype(str), classes[blend_test.argmax(1)].astype(str)))
submission = pd.DataFrame({"id": sample_sub["post_id"].astype(str),
                           "label": sample_sub["post_id"].astype(str).map(pred)})
assert len(submission) == 1440 and submission["label"].notna().all()
submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)
print("Saved", OUTPUT_DIR / "submission.csv")